## 1. Importação de bibliotecas

In [20]:
!pip install transformers torch accelerate -q
import random
import kagglehub
import csv
import pandas as pd
import numpy as np
import torch
import os
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib
import re
from transformers import AutoModelForCausalLM, AutoTokenizer # AutoTokenizer e o AutoModelForCausalLM automatizam o processo de importação do modelo, puxando os dados e informações necessárias diretamente do Hugging Face

## 2. Dados
Os dados foram gerados tentando manter uma proporção 33%/33%/33% com base no tipo de imóvel.

In [3]:
"""
Gerador de base de dados sintética para o problema de análise energética.
Gera apenas as colunas de entrada (features), com consumo_kwh correlacionado
de forma realista ao tipo de imóvel, quantidade de equipamentos e horas de
alto consumo, em vez de valores puramente aleatórios.
"""

TIPOS_IMOVEL = ["Casa", "Apartamento", "Comercial"]
TARIFA_KWH = 0.75


def gerar_registro(id_cliente):
    tipo_imovel = random.choice(TIPOS_IMOVEL)
    quantidade_equipamentos = random.randint(1, 25)
    horas_alto_consumo = round(random.uniform(0, 12), 1)
    uso_horario_pico = random.random() < 0.5

    # Consumo base varia conforme tipo de imóvel
    base_por_tipo = {
        "Casa": 250,
        "Apartamento": 150,
        "Comercial": 500,
    }[tipo_imovel]

    # Consumo cresce com equipamentos e horas de alto consumo, com ruído aleatório
    consumo_kwh = (
        base_por_tipo
        + quantidade_equipamentos * random.uniform(8, 15)
        + horas_alto_consumo * random.uniform(10, 20)
        + (50 if uso_horario_pico else 0)
        + random.gauss(0, 30)  # ruído
    )
    consumo_kwh = max(20, round(consumo_kwh, 1))

    return {
        "id_cliente": id_cliente,
        "consumo_kwh": consumo_kwh,
        "uso_horario_pico": uso_horario_pico,
        "quantidade_equipamentos": quantidade_equipamentos,
        "tipo_imovel": tipo_imovel,
        "horas_alto_consumo": horas_alto_consumo,
    }


def gerar_base(n_registros=90000, caminho_saida="base_energetica.csv"):
    registros = [gerar_registro(id_cliente=i) for i in range(1, n_registros + 1)]

    with open(caminho_saida, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=registros[0].keys())
        writer.writeheader()
        writer.writerows(registros)

    print(f"Base gerada com {n_registros} registros em '{caminho_saida}'")
    return registros


if __name__ == "__main__":
    gerar_base(n_registros=90000, caminho_saida="base_energetica.csv")

Base gerada com 90000 registros em 'base_energetica.csv'


In [4]:
df = pd.read_csv("/content/base_energetica.csv")

Kaggle

In [5]:
caminho = kagglehub.dataset_download("samxsam/household-energy-consumption")
print("Arquivos baixados:", os.listdir(caminho))

nome_arquivo = os.listdir(caminho)[0]
df_kaggle = pd.read_csv(os.path.join(caminho, nome_arquivo))

print(df_kaggle.shape)
print(df_kaggle.columns.tolist())
df_kaggle.head()

100%|██████████| 731k/731k [00:00<00:00, 93.2MB/s]

Extracting files...
Arquivos baixados: ['household_energy_consumption.csv']
(90000, 7)
['Household_ID', 'Date', 'Energy_Consumption_kWh', 'Household_Size', 'Avg_Temperature_C', 'Has_AC', 'Peak_Hours_Usage_kWh']


,Household_ID,Date,Energy_Consumption_kWh,Household_Size,Avg_Temperature_C,Has_AC,Peak_Hours_Usage_kWh
0,H00001,2025-04-01,8.4,4,17.8,No,3.2
1,H00001,2025-04-02,7.9,4,17.3,No,2.8
2,H00001,2025-04-03,9.2,4,18.6,No,3.0
3,H00001,2025-04-04,7.9,4,18.2,No,2.7
4,H00001,2025-04-05,9.6,4,11.9,No,3.2


In [6]:
df['ar_condicionado'] = df_kaggle['Has_AC']
df["uso_horario_pico"] = df["uso_horario_pico"].astype(bool)
df["ar_condicionado"] = df["ar_condicionado"].map({"Yes": True, "No": False})
df.to_csv("base_energetica.csv", index=False)
df.head()

,id_cliente,consumo_kwh,uso_horario_pico,quantidade_equipamentos,tipo_imovel,horas_alto_consumo,ar_condicionado
0,1,636.9,True,24,Apartamento,7.8,False
1,2,858.2,False,23,Comercial,4.6,False
2,3,801.8,True,23,Casa,9.7,False
3,4,486.3,True,20,Apartamento,7.2,False
4,5,436.5,False,7,Casa,6.6,False


## 3. Exploração e limpeza dos dados

In [7]:
print("Dimensões:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)
print("\nValores nulos por coluna:")
print(df.isnull().sum())
print("\nRegistros duplicados:", df.duplicated().sum())

print("\nInformações adicionais:")
q33 = np.percentile(df['consumo_kwh'], 33)
q66 = np.percentile(df['consumo_kwh'], 66)
consumo_minimo = df['consumo_kwh'].min()
consumo_maximo = df['consumo_kwh'].max()
print('Consumo Minimo: ', consumo_minimo)
print('Consumo Maximo: ', consumo_maximo)
print('Percentil 33: ', q33)
print('Percentil 66: ', q66)

df.head().style.hide(axis="index")

Dimensões: (90000, 7)

Tipos de dados:
id_cliente                   int64
consumo_kwh                float64
uso_horario_pico              bool
quantidade_equipamentos      int64
tipo_imovel                 object
horas_alto_consumo         float64
ar_condicionado               bool
dtype: object

Valores nulos por coluna:
id_cliente                 0
consumo_kwh                0
uso_horario_pico           0
quantidade_equipamentos    0
tipo_imovel                0
horas_alto_consumo         0
ar_condicionado            0
dtype: int64

Registros duplicados: 0

Informações adicionais:
Consumo Minimo:  93.1
Consumo Maximo:  1167.0
Percentil 33:  459.8
Percentil 66:  640.6


id_cliente,consumo_kwh,uso_horario_pico,quantidade_equipamentos,tipo_imovel,horas_alto_consumo,ar_condicionado
1,636.900000,True,24,Apartamento,7.800000,False
2,858.200000,False,23,Comercial,4.600000,False
3,801.800000,True,23,Casa,9.700000,False
4,486.300000,True,20,Apartamento,7.200000,False
5,436.500000,False,7,Casa,6.600000,False


In [8]:
# Variáveis numéricas
print("Estatísticas descritivas")
display(df[["consumo_kwh", "quantidade_equipamentos", "horas_alto_consumo"]].describe())

# Variáveis categóricas/booleanas
print("\nDistribuição de tipo_imovel")
print(df["tipo_imovel"].value_counts())
print("\nDistribuição de uso_horario_pico")
print(df["uso_horario_pico"].value_counts(normalize=False))

Estatísticas descritivas


,consumo_kwh,quantidade_equipamentos,horas_alto_consumo
count,90000.000000,90000.000000,90000.000000
mean,564.347848,12.989800,5.992428
std,184.016828,7.217958,3.459588
min,93.100000,1.000000,0.000000
25%,421.300000,7.000000,3.000000
50%,545.300000,13.000000,6.000000
75%,701.100000,19.000000,9.000000
max,1167.000000,25.000000,12.000000



Distribuição de tipo_imovel
tipo_imovel
Casa           30108
Comercial      29973
Apartamento    29919
Name: count, dtype: int64

Distribuição de uso_horario_pico
uso_horario_pico
False    45201
True     44799
Name: count, dtype: int64


## 4. Definição de categorias

In [9]:
"""
1. Rotula a base sintética criando um índice de ineficiência energética
   e cortando-o em quintis (Excelente / Bom / Mediano / Ruim / Crítico).
2. Treina e compara três classificadores: Regressão Logística, Random Forest
   e Árvore de Decisão.
3. Salva o melhor modelo (pipeline completo com pré-processamento).
"""

CAMINHO_ENTRADA = "base_energetica.csv"
CAMINHO_SAIDA = "base_energetica_rotulada.csv"

COLUNAS_NUMERICAS = ["consumo_kwh", "quantidade_equipamentos", "horas_alto_consumo"]
COLUNAS_CATEGORICAS = ["tipo_imovel"]
COLUNAS_BOOLEANAS = ["uso_horario_pico", "ar_condicionado"]

# Consumo médio de referência por tipo de imóvel (mesma lógica usada na geração da base)
CONSUMO_BASE_POR_TIPO = {
    "Casa": 250,
    "Apartamento": 150,
    "Comercial": 500,
    "Industrial": 1200,
}


def calcular_indice_ineficiencia(df):
    """Índice composto (0 a 1) combinando as variáveis de entrada disponíveis."""
    consumo_relativo = df["consumo_kwh"] / df["tipo_imovel"].map(CONSUMO_BASE_POR_TIPO)
    consumo_norm = (consumo_relativo / consumo_relativo.max()).clip(0, 1)
    equip_norm = (df["quantidade_equipamentos"] / df["quantidade_equipamentos"].max()).clip(0, 1)
    horas_norm = (df["horas_alto_consumo"] / df["horas_alto_consumo"].max()).clip(0, 1)
    pico_norm = df["uso_horario_pico"].astype(int)
    ac_norm = df["ar_condicionado"].astype(int)

    indice = (
        0.35 * consumo_norm
        + 0.20 * pico_norm
        + 0.20 * equip_norm
        + 0.15 * horas_norm
        + 0.10 * ac_norm
    )
    return indice


def rotular_por_quintis(indice):
    """Corta o índice em 5 faixas com quantidades iguais de cada classe."""
    return pd.qcut(
        indice,
        q=5,
        labels=["Excelente", "Bom", "Mediano", "Ruim", "Crítico"]
    )


def main():
    df = pd.read_csv(CAMINHO_ENTRADA)
    df["uso_horario_pico"] = df["uso_horario_pico"].astype(int)
    df["ar_condicionado"] = df["ar_condicionado"].astype(int)

    indice = calcular_indice_ineficiencia(df)
    df["categoria"] = rotular_por_quintis(indice)
    df.to_csv(CAMINHO_SAIDA, index=False)

    print("Distribuição das categorias:")
    print(df["categoria"].value_counts(), "\n")


if __name__ == "__main__":
    main()

Distribuição das categorias:
categoria
Excelente    18000
Bom          18000
Mediano      18000
Ruim         18000
Crítico      18000
Name: count, dtype: int64 



In [10]:
df = pd.read_csv("/content/base_energetica_rotulada.csv")
df.head()

,id_cliente,consumo_kwh,uso_horario_pico,quantidade_equipamentos,tipo_imovel,horas_alto_consumo,ar_condicionado,categoria
0,1,636.9,1,24,Apartamento,7.8,0,Crítico
1,2,858.2,0,23,Comercial,4.6,0,Bom
2,3,801.8,1,23,Casa,9.7,0,Crítico
3,4,486.3,1,20,Apartamento,7.2,0,Crítico
4,5,436.5,0,7,Casa,6.6,0,Excelente


In [11]:
df['estimativa_financeira'] = df['consumo_kwh'] * TARIFA_KWH
df['consumo_kwh'] = df['consumo_kwh'].round(1)
df.head()

,id_cliente,consumo_kwh,uso_horario_pico,quantidade_equipamentos,tipo_imovel,horas_alto_consumo,ar_condicionado,categoria,estimativa_financeira
0,1,636.9,1,24,Apartamento,7.8,0,Crítico,477.675
1,2,858.2,0,23,Comercial,4.6,0,Bom,643.650
2,3,801.8,1,23,Casa,9.7,0,Crítico,601.350
3,4,486.3,1,20,Apartamento,7.2,0,Crítico,364.725
4,5,436.5,0,7,Casa,6.6,0,Excelente,327.375


In [12]:
df['ar_condicionado'].value_counts()

,count
ar_condicionado,
0,45508
1,44492


## 5. Treinamento do modelo de classificação

In [13]:
COLUNAS_NUMERICAS = ["consumo_kwh", "quantidade_equipamentos", "horas_alto_consumo"]
COLUNAS_CATEGORICAS = ["tipo_imovel"]
COLUNAS_BOOLEANAS = ["uso_horario_pico", "ar_condicionado"]

pre_processador = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), COLUNAS_NUMERICAS + COLUNAS_BOOLEANAS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), COLUNAS_CATEGORICAS),
    ]
)

pipeline = Pipeline([
    ("pre", pre_processador),
    ("modelo", LogisticRegression(max_iter=1000)),
])

X = df[COLUNAS_NUMERICAS + COLUNAS_CATEGORICAS + COLUNAS_BOOLEANAS]
y = df["categoria"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline.fit(X_treino, y_treino)
y_pred = pipeline.predict(X_teste)

print(f"Acurácia: {accuracy_score(y_teste, y_pred):.3f}\n")
print(classification_report(y_teste, y_pred, zero_division=0))

joblib.dump(pipeline, "modelo_categorizacao.joblib")
print("\nModelo salvo em 'modelo_categorizacao.joblib'")

Acurácia: 0.948

              precision    recall  f1-score   support

         Bom       0.93      0.93      0.93      3600
     Crítico       0.98      0.97      0.97      3600
   Excelente       0.97      0.97      0.97      3600
     Mediano       0.92      0.93      0.92      3600
        Ruim       0.94      0.94      0.94      3600

    accuracy                           0.95     18000
   macro avg       0.95      0.95      0.95     18000
weighted avg       0.95      0.95      0.95     18000


Modelo salvo em 'modelo_categorizacao.joblib'


## 6. Importação do modelo para as recomendações

In [14]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print(f"Modelo carregado em: {model.device}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado em: cuda:0


In [21]:
# Função geradora de recomendações
def gerar_recomendacoes(df: dict, categoria: str, max_new_tokens: int = 200) -> list[str]:
    """
    Gera 3 recomendações de eficiência energética com base nos dados de entrada
    e na categoria já calculada pelo classificador (Regressão Logística).
    """
    system_prompt = (
        "Você é um assistente especializado em eficiência energética residencial e comercial. "
        "Responda sempre em português do Brasil, de forma objetiva e sem rodeios."
    )

    regra_ac = (
        "O imóvel POSSUI ar-condicionado — pode mencioná-lo se for relevante."
        if df['ar_condicionado']
        else "O imóvel NÃO possui ar-condicionado — NUNCA mencione ar-condicionado em nenhuma recomendação."
    )

    regra_tom = (
        "Tom conforme a categoria: Excelente ou Bom = reforçar boas práticas já adotadas; "
        "Mediano = sugerir ajustes pontuais; Ruim ou Crítico = ser direto sobre a necessidade "
        "de mudança, com mais urgência em Crítico."
    )

    user_prompt = f"""Com base nos dados abaixo, gere exatamente 3 recomendações curtas, práticas
e realmente úteis para melhorar a eficiência energética do imóvel.

REGRAS OBRIGATÓRIAS:
- Envolva EXCLUSIVAMENTE: hábitos de uso de equipamentos elétricos, horários de consumo, ou
  manutenção/substituição de aparelhos elétricos. Nada de água, gás ou outros recursos.
- {regra_ac}
- Baseie-se APENAS nos dados fornecidos. Não invente equipamentos ou hábitos não informados.
- Se o tipo de imóvel for "Apartamento", não sugira painéis solares ou soluções que dependam
  de telhado/espaço externo próprio.
- Cada recomendação deve abordar um aspecto diferente, sem repetir o mesmo tipo de dica.
- Não cite marcas, modelos ou preços. Não use termos técnicos sem explicação simples.
- Máximo 20 palavras por recomendação. Sem emojis, markdown ou numeração.
- {regra_tom}

Exemplo de estilo (não copie o conteúdo):
Reduzir o uso simultâneo de equipamentos durante o horário de pico.
Desligar aparelhos em modo stand-by quando não estiverem em uso.
Distribuir o uso de equipamentos de maior consumo ao longo do dia.

Dados do imóvel:
- Consumo mensal: {df['consumo_kwh']} kWh
- Uso em horário de pico: {"Sim" if df['uso_horario_pico'] else "Não"}
- Quantidade de equipamentos: {df['quantidade_equipamentos']}
- Tipo de imóvel: {df['tipo_imovel']}
- Horas de alto consumo por dia: {df['horas_alto_consumo']}
- Categoria de eficiência: {categoria}
- Possui ar-condicionado: {"Sim" if df['ar_condicionado'] else "Não"}

Responda APENAS com as 3 recomendações, uma por linha, sem numeração,
sem introdução e sem comentários adicionais."""

    mensagens = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    texto_prompt = tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True
    )
    model_inputs = tokenizer([texto_prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # geração determinística — evita "viagens" do modelo
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    resposta = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    recomendacoes = [
        re.sub(r"^\s*[\d]+[\.\)]?\s*", "", linha).strip("-•* ").strip()
        for linha in resposta.strip().split("\n")
        if linha.strip()
    ]
    return recomendacoes[:3]

In [22]:
modelo = joblib.load("modelo_categorizacao.joblib")

cliente_id = random.randint(1, len(df))
linha_cliente = df[df["id_cliente"] == cliente_id].iloc[0]
dados_cliente = linha_cliente[["consumo_kwh", "uso_horario_pico", "quantidade_equipamentos", "tipo_imovel", "horas_alto_consumo", "ar_condicionado"]].to_dict()

# Agora a categoria vem do MODELO, não do índice de tercis
dados_cliente_df = pd.DataFrame([dados_cliente])
categoria = modelo.predict(dados_cliente_df)[0]
probabilidade = round(modelo.predict_proba(dados_cliente_df).max(), 2)

recomendacoes = gerar_recomendacoes(dados_cliente, categoria)

In [23]:
print(linha_cliente)
print(recomendacoes)
print(f"Categoria prevista: {categoria} (confiança: {probabilidade})")

id_cliente                       66742
consumo_kwh                      468.6
uso_horario_pico                     1
quantidade_equipamentos              6
tipo_imovel                Apartamento
horas_alto_consumo                11.8
ar_condicionado                      0
categoria                         Ruim
estimativa_financeira           351.45
Name: 66741, dtype: object
['Desligue os aparelhos em modo stand-by quando não estiverem em uso.', 'Distribua o uso dos equipamentos de maior consumo ao longo do dia.', 'Reduza o uso simultâneo de equipamentos durante o horário de pico.']
Categoria prevista: Ruim (confiança: 0.87)


### Exemplo saída comercial:
['1. Instale painéis solares fotovoltaicos.', '2. Faça um planejamento térmico da fachada do edifício.', '3. Optimize o gerenciamento da iluminação e das temperaturas interna.']

### Exemplo saída residencial (Casa)
['1. Reduza o número de equipamentos elétricos instalados.', '2. Implemente um plano de programação da luz para minimizar o uso na alta demanda.', '3. Opte por fontes alternativas de energia no fornecimento doméstico.']

### Exemplo de saída residencial (Apartamento)
id_cliente                         605
consumo_kwh                      289.1
uso_horario_pico                     1
**quantidade_equipamentos              1**
tipo_imovel                Apartamento
horas_alto_consumo                10.7
categoria                     Moderado
estimativa_financeira          216.825
Name: 604, dtype: object

['1. Desligue o equipamento principal em modo stand-by quando não estiver em uso.', '2. Distribua o uso do equipamento entre diferentes dias da semana para evitar o horário de pico.', '3. Faça uma análise detalhada do consumo diário para identificar áreas onde pode reduzir o uso excessivo.']


